# EA722 – Laboratório de Controle e Servomecanismo

## Experiência 7: Preditores e Controladores Não-Lineares

### Parte 3: Linear Predictor, MLP, CNN, Simple Recurrent Neural Network, LSTM, and GRU for multi-step time series prediction.

Universidade Estadual de Campinas – UNICAMP <br>
Faculdade de Engenharia Elétrica e de Computação – FEEC <br>

**Professores:** Fernando J. Von Zuben / Caíque Santos Lima <br>
**Grupo / Bancada:** T1, T2, R1, R2 ou E <br>
**Turma:** K, L, U ou V <br>
**Aluno(a):** , **RA:** ?????? <br>
**Aluno(a):** , **RA:** ?????? <br>
**Aluno(a):** , **RA:** ?????? <br>

In [ ]:
# Loading all the relevant libraries
from pandas import read_csv
from pandas import DataFrame
import numpy as np
import pandas as pd
import tensorflow as tf
from keras.models import Sequential
from keras.layers import Dense, SimpleRNN, LSTM, GRU, Conv1D, MaxPooling1D, Flatten, Input
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error
import math
import matplotlib.pyplot as plt

#### **Please, upload the datasets [monthly_sunspot.csv] and [TS.csv] before executing the code. The datasets are provided together with this code. The dataset [monthly_sunspot.csv] is available at [this link](https://github.com/jbrownlee/Datasets/blob/master/monthly-sunspots.csv). The dataset [TS.csv] is a synthetic nonlinear time series designed specifically for this activity.**

In [ ]:
# Structuring the dataset from a single time series, scaling and spliting into training / testing
def get_train_test(dataset, split_percent, time_steps, horizon):
    df = read_csv(dataset, usecols=[1], engine='python')
    data = np.array(df.values.astype('float32'))
    scaler = MinMaxScaler(feature_range=(0, 1))
    # There is a kind of data leakage in the next command
    data = scaler.fit_transform(data).flatten()
    n = len(data)
    df = DataFrame()
    for i in range(0,time_steps):
      df['t-' + str(time_steps-i-1)] = [data[j] for j in range(i, (len(data)-horizon-time_steps+i+1))]
    for i in range(1,horizon+1):
      df['t+' + str(i)] = [data[j] for j in range(time_steps+i-1, (len(data)-horizon+i))]
    print(df)
    df1 = df.copy()
    for i in range(1,horizon+1):
      df1 = df1.drop(['t+' + str(i)], axis = 1)
    dataX = np.array(df1) #separacao de dados de time_steps
    df2 = df['t+1']
    for i in range(2,horizon+1):
      df2 = pd.concat([df2, df['t+' + str(i)]], axis=1)
    datay = np.array(df2) #separacao de dados de horizon
    # Point for splitting data into train and test
    split = int(n*split_percent)
    X = dataX[range(split),:]
    y = datay[range(split)]
    Xt = dataX[split:,:]
    yt = datay[split:]
    return X, y, Xt, yt

time_steps = 22
horizon = 3
# Here you will decide which dataset to consider.
X, y, Xt, yt = get_train_test('TS.csv', 0.8, time_steps, horizon)
# X, y, Xt, yt = get_train_test('monthly_sunspot.csv', 0.8, time_steps, horizon)

#### **Please, to switch between the two datasets, comment the appropriate command at the end of the code cell above and uncomment the other.**

In [ ]:
print(X.shape)
print(y.shape)

For a definition of the autocorrelation function (ACF) and of the partial autocorrelation function (PACF), please refer to [this content](https://en.wikipedia.org/wiki/Autocorrelation) and also to [this content](https://en.wikipedia.org/wiki/Partial_autocorrelation_function).

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Be coherent with your decision 2 cells above
df = read_csv('TS.csv', usecols=[1], engine='python')
# df = read_csv('monthly_sunspot.csv', usecols=[1], engine='python')
data = np.array(df.values.astype('float32'))
plot_acf(data)
plot_pacf(data)
plt.show()

## **Implementing a linear predictor**

In [ ]:
def linear_predictor(X,y):
    model = LinearRegression()
    train_predict = model.fit(X, y)
    print(model.coef_)
    print(model.intercept_)

    return model

linear_model = linear_predictor(X,y)

## **Implementing the multilayer perceptron neural network (MLP)**

In [ ]:
def MLP_predictor():
  model = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(horizon, activation='linear')
  ])
  model.compile(optimizer='adam', loss='mean_squared_error')
  train_predict = model.fit(X, y, epochs=100)
  model.summary()

  return model

MLP_model = MLP_predictor()

## **Implementing the convolutional neural network (CNN)**

In [ ]:
def CNN_predictor():
  model = tf.keras.models.Sequential([
    tf.keras.layers.Input(shape=(time_steps, 1)),
    tf.keras.layers.Conv1D(filters=128, kernel_size=2, activation='relu'),
    tf.keras.layers.MaxPooling1D(pool_size=2),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(100, activation='relu'),
    tf.keras.layers.Dense(horizon, activation='linear')
  ])
  model.compile(optimizer='adam', loss='mean_squared_error')
  train_predict = model.fit(X, y, epochs=100)
  model.summary()

  return model

CNN_model = CNN_predictor()

## **Implementing the simple recurrent neural network**

In [ ]:
def create_RNN(hidden_units, dense_units, output_units, input_shape, activation):
        model = Sequential()
        model.add(Input(shape=input_shape))
        model.add(SimpleRNN(hidden_units, activation=activation[0]))
        model.add(Dense(units=dense_units, activation=activation[2]))
        model.add(Dense(units=output_units, activation=activation[1]))
        model.compile(loss='mean_squared_error', optimizer='adam')
        return model

def RNN_predictor (X, y, time_steps):
    model = create_RNN(hidden_units=30, dense_units = 50, output_units=horizon, input_shape=(time_steps,1), activation=['tanh', 'linear','relu'])
    train_predict = model.fit(X, y, epochs=40, batch_size=1, verbose=2)
    model.summary()

    return model

RNN_model = RNN_predictor(X, y, time_steps)

In [ ]:
# [Do not run this cell] or [Upload the PNG file provided together with the code and then run].
# Configuration of the simple RNN
from IPython.display import Image
Image("RNN_info_flow.png", width = 600, height = 300)

In [ ]:
wx = RNN_model.get_weights()[0]
wh = RNN_model.get_weights()[1]
bh = RNN_model.get_weights()[2]
wy1 = RNN_model.get_weights()[3]
by1 = RNN_model.get_weights()[4]
wy2 = RNN_model.get_weights()[5]
by2 = RNN_model.get_weights()[6]
print(wx.shape)
print(wh.shape)
print(bh.shape)
print(wy1.shape)
print(by1.shape)
print(wy2.shape)
print(by2.shape)

## **Implementing the long short-term memory network (LSTM)**

In [ ]:
def LSTM_predictor(X, y, time_steps):
    X1 = X.reshape(X.shape[0],time_steps,1)
    model=Sequential()
    model.add(Input(shape=(time_steps,1)))
    model.add(LSTM(30))
    model.add(Dense(50,activation='relu'))
    model.add(Dense(horizon,activation='linear'))
    model.compile(loss='mean_squared_error',optimizer='adam')

    train_predict = model.fit(X1,y,epochs=100,verbose=2)

    return model

LSTM_model = LSTM_predictor(X, y, time_steps)
print(LSTM_model.summary())

## **Implementing the Gated Recurrent Unit (GRU)**

In [ ]:
def GRU_predictor(X, y, time_steps):
    X1 = X.reshape(X.shape[0],time_steps,1)
    model=Sequential()
    model.add(Input(shape=(time_steps,1)))
    model.add(GRU(30))
    model.add(Dense(50,activation='relu'))
    model.add(Dense(horizon,activation='linear'))
    model.compile(loss='mean_squared_error',optimizer='adam')

    train_predict = model.fit(X1,y,epochs=100,verbose=2)

    return model

GRU_model = GRU_predictor(X, y, time_steps)
print(GRU_model.summary())

In [ ]:
def print_error(trainY, testY, train_predict, test_predict,i):
    # Error of predictions
    train_rmse = math.sqrt(mean_squared_error(trainY, train_predict))
    test_rmse = math.sqrt(mean_squared_error(testY, test_predict))
    # Print RMSE
    print('t+%d RMSE train: %.4f' % (i+1, train_rmse))
    print('t+%d RMSE test: %.4f' % (i+1, test_rmse))
    return [train_rmse, test_rmse]


def RMSE(train_predict, test_predict):
  rmse = []
  for i in range(horizon):
    if(horizon == 1):
      y1_pred = train_predict
      y1_pred_t = test_predict
      y1 = y
      y1_t = yt
    elif(horizon > 1):
      y1_pred = train_predict[:,i]
      y1_pred_t = test_predict[:,i]
      y1 = y[:,i]
      y1_t = yt[:,i]

    erro = print_error(y1, y1_t, y1_pred, y1_pred_t,i)
    rmse.append(erro)

  # Create a DataFrame from the rmse values for each horizon
  df = pd.DataFrame(rmse)
  df.columns = ['train', 'test']
  return df


In [ ]:
# Plot the predictions together with the actual values
def plot_result(trainY, testY, train_predict, test_predict,i):
    actual = np.append(trainY, testY)
    predictions = np.append(train_predict, test_predict)
    rows = len(actual)
    plt.figure(figsize=(30, 6), dpi=300)
    plt.plot(range(rows), actual)
    plt.plot(range(rows), predictions)
    plt.axvline(x=len(trainY), color='r')
    plt.legend(['Actual', 'Predictions'])
    plt.xlabel('Time steps')
    plt.ylabel('Actual and Predicted Values')
    plt.title('t+%d Predition (The Red Line Separates The Training And Test Examples)' % (i+1))
    plt.grid(True)

def plot_predict(train_predict, test_predict, y, yt):
  for i in range(horizon):
    if(horizon == 1):
      y1_pred = train_predict
      y1_pred_t = test_predict
      y1 = y
      y1_t = yt
    elif(horizon > 1):
      y1_pred = train_predict[:,i]
      y1_pred_t = test_predict[:,i]
      y1 = y[:,i]
      y1_t = yt[:,i]
    plot_result(y1, y1_t, y1_pred, y1_pred_t,i)


In [ ]:
print ('***** LINEAR MODEL *****')
train_predict = linear_model.predict(X)
test_predict = linear_model.predict(Xt)
df1 = RMSE(train_predict, test_predict)
plot_predict(train_predict, test_predict, y, yt)

In [ ]:
print ('***** MLP MODEL *****')
train_predict = MLP_model.predict(X)
test_predict = MLP_model.predict(Xt)
df2 = RMSE(train_predict, test_predict)
plot_predict(train_predict, test_predict, y, yt)

In [ ]:
print ('***** CNN MODEL *****')
train_predict = CNN_model.predict(X)
test_predict = CNN_model.predict(Xt)
df3 = RMSE(train_predict, test_predict)
plot_predict(train_predict, test_predict, y, yt)

In [ ]:
print ('***** RNN MODEL *****')
train_predict = RNN_model.predict(X)
test_predict = RNN_model.predict(Xt)
df4 = RMSE(train_predict, test_predict)
plot_predict(train_predict, test_predict, y, yt)

In [ ]:
print ('***** LSTM MODEL *****')
train_predict = LSTM_model.predict(X)
test_predict = LSTM_model.predict(Xt)
df5 = RMSE(train_predict, test_predict)
plot_predict(train_predict, test_predict, y, yt)

In [ ]:
print ('***** GRU MODEL *****')
train_predict = GRU_model.predict(X)
test_predict = GRU_model.predict(Xt)
df6 = RMSE(train_predict, test_predict)
plot_predict(train_predict, test_predict, y, yt)

In [ ]:
# Plot the prediction error
def plot_result2(trainY, testY, train_predict, test_predict,i):
    actual = np.append(trainY, testY)
    predictions = np.append(train_predict, test_predict)
    rows = len(actual)
    plt.figure(figsize=(30, 6), dpi=300)
    plt.plot(range(rows), actual - predictions)
    plt.axvline(x=len(trainY), color='r')
    plt.xlabel('Time steps')
    plt.ylabel('Actual minus Predicted Values')
    plt.title('t+%d Predition Error (The Red Line Separates The Training And Test Examples)' % (i+1))
    plt.ylim(-0.5, +1.0) # Please, set the interval appropriately, considering all the graphs.
    plt.grid(True)

def plot_error_predict(train_predict, test_predict, y, yt):
    for i in range(horizon):
      if(horizon == 1):
        y1_pred = train_predict
        y1_pred_t = test_predict
        y1 = y
        y1_t = yt
      elif(horizon > 1):
        y1_pred = train_predict[:,i]
        y1_pred_t = test_predict[:,i]
        y1 = y[:,i]
        y1_t = yt[:,i]
      plot_result2(y1, y1_t, y1_pred, y1_pred_t,i)

In [ ]:
# We are exhibiting the error produced by a single prediction model, just to illustrate the general behavior.
print ('***** LINEAR MODEL *****')
train_predict = linear_model.predict(X)
test_predict = linear_model.predict(Xt)
plot_error_predict(train_predict, test_predict, y, yt)

## **Exhibit a dataframe with the final results for each predictor**

In [ ]:
DF_results = pd.concat([df1, df2, df3, df4, df5, df6], axis=1, keys=['LINEAR', 'MLP', 'CNN', 'RNN', 'LSTM', 'GRU'])
DF_results.index = ['t+1', 't+2', 't+3']
DF_results

<font color="green">
Atividade 3.1 <br>
Explique o papel dos parâmetros de projeto [time_steps] e [horizon].
</font>

Resposta:

<font color="green">
Atividade 3.2 <br>
Qual a diferença no uso do parâmetro [time_steps] no caso dos preditores não-recorrentes e recorrentes?
</font>

Resposta:

<font color="green">
Atividade 3.3 <br>
No caso do preditor Simple Recurrent Neural Network (RNN), se baseie na notação da figura apresentada e explique as dimensões envolvidas. Repare que foram consideradas, ao final, duas camadas densas e não apenas uma, como na figura.
</font>

Resposta:

<font color="green">
Atividade 3.4 <br>
Qual modelo de predição levou a um melhor desempenho junto aos dados de teste em cada caso?
</font>

Resposta:

<font color="green">
Atividade 3.5 <br>
Apresente uma hipótese para explicar o motivo pelo qual todos os preditores sofrem uma degradação de desempenho (não compare os preditores entre si, mas sim cada preditor para os 3 passos à frente de predição) conforme a predição envolve um futuro mais distante, independente do caso de estudo.
</font>

Resposta: